In [1]:
# Make this notebook work from either fine-tuning/ or fine-tuning/pre-demo/
# (idempotent: re-running is safe)
import os
from pathlib import Path
_here = Path.cwd()
if _here.name in ('pre-demo', 'live-demo'):
    os.chdir(_here.parent)
print('cwd:', Path.cwd())


cwd: C:\Users\okofoworola\Acme Health Demo\fine-tuning


# Lab 11 · Agents & Tool Orchestration (Foundry Agent Service)

A single fine-tuned model answers one turn. A real Member Services call is **multi-step**: verify identity → look up prescriptions → check the formulary → request the refill. This lab promotes the model into a **persistent Foundry Agent** that orchestrates those tools itself — the platform plans the steps, calls your functions, and threads the conversation. *This is the capability most teams can't build alone, and Foundry gives it to you as a managed service.*


---
## Step 0 — Enable Foundry tracing

*Wire OpenTelemetry to Application Insights so every model call below shows up live in the Microsoft Foundry portal under **your project → Tracing**.*


In [2]:
from dotenv import load_dotenv
load_dotenv(override=True)

import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve()))
from _tracing import enable_foundry_tracing

enable_foundry_tracing(service_name='acme-agents-lab')


[tracing] enabled. Service name : acme-agents-lab
[tracing] Open Foundry portal -> your project -> Tracing tab
[tracing] (also visible under App Insights: appi-shuttervoice-dev)


True

---
## Step 1 — Connect to the Foundry Agent Service (LIVE)

Your project (`agents`) already hosts agents — see `_seed_grounded_tools_agent.py`. We connect with the same managed identity / `az login` credential. Degrades gracefully if the SDK or the `Azure AI Developer` role isn't present.


In [3]:
import os
from _advisor import get_credential

ACCT    = os.environ.get('AZURE_RESOURCE_NAME', 'aif-shuttervoice-dev')
PROJECT = os.environ.get('AZURE_FOUNDRY_PROJECT', 'agents')
PROJECT_ENDPOINT = f'https://{ACCT}.services.ai.azure.com/api/projects/{PROJECT}'

agents = None
try:
    from azure.ai.agents import AgentsClient
    agents = AgentsClient(endpoint=PROJECT_ENDPOINT, credential=get_credential())
    print('Agents client ready ->', PROJECT_ENDPOINT)
except Exception as e:
    print('[skip] Foundry Agent Service unavailable:', type(e).__name__, e)


Agents client ready -> https://aif-shuttervoice-dev.services.ai.azure.com/api/projects/agents


---
## Step 2 — Register an agent with Acme tools

We give the agent three Member Services functions and let Foundry decide when to call each. The functions run **locally** (your business logic); the agent only orchestrates.


In [4]:
import json

# --- local business logic (stubbed; wire to real systems in prod) ---
def verify_member_identity(member_id: str, dob: str) -> str:
    return json.dumps({'member_id': member_id, 'verified': True})

def lookup_prescriptions(member_id: str) -> str:
    return json.dumps({'prescriptions': [
        {'name': 'lisinopril 10mg', 'refills_left': 2, 'status': 'active'},
        {'name': 'atorvastatin 20mg', 'refills_left': 0, 'status': 'expired'}]})

def request_refill(member_id: str, drug_name: str) -> str:
    return json.dumps({'drug': drug_name, 'refill': 'submitted', 'eta_days': 2})

SYSTEM = ('You are a Acme Health Member Services agent. Verify identity '
          'before disclosing PHI, then help with prescriptions and refills. '
          'Use tools; never invent prescription data.')

agent = thread = None
if agents is not None:
    try:
        from azure.ai.agents.models import FunctionTool, ToolSet
        toolset = ToolSet()
        toolset.add(FunctionTool(functions={verify_member_identity,
                                            lookup_prescriptions,
                                            request_refill}))
        agents.enable_auto_function_calls(toolset)
        agent = agents.create_agent(model=os.environ.get('BASE_DEPLOYMENT', 'gpt-4o-mini'),
                                    name='acme-orchestration-demo',
                                    instructions=SYSTEM, toolset=toolset)
        print('Agent created:', agent.id)
    except Exception as e:
        print('[skip] could not create agent:', type(e).__name__, e)
else:
    print('[skip] no agents client - see Step 1.')


Agent created: asst_wc9V4vbBb5WsA2otwwIqxPOG


---
## Step 3 — Run a multi-step conversation

One member message that requires **three** tool calls in order. Watch Foundry plan and execute the chain, then read the run steps to see the orchestration trace.


In [5]:
if agent is not None:
    try:
        thread = agents.threads.create()
        agents.messages.create(thread_id=thread.id, role='user', content=(
            'Hi, this is member M-10293, DOB 04/12/1958. '
            'Can you refill my expired cholesterol medication?'))
        run = agents.runs.create_and_process(thread_id=thread.id, agent_id=agent.id)
        print('Run status:', run.status)

        steps = agents.run_steps.list(thread_id=thread.id, run_id=run.id)
        for s in steps:
            det = getattr(s, 'step_details', None)
            kind = getattr(det, 'type', '?')
            print('  step:', getattr(s, 'type', '?'), '->', kind)

        for m in agents.messages.list(thread_id=thread.id):
            if m.role == 'assistant':
                for c in m.content:
                    if getattr(c, 'text', None):
                        print('\nASSISTANT:', c.text.value)
                break
    except Exception as e:
        print('[skip] run failed:', type(e).__name__, e)
else:
    print('[skip] no agent - see Step 2.')


Run status: RunStatus.COMPLETED


  step: RunStepType.MESSAGE_CREATION -> message_creation
  step: RunStepType.TOOL_CALLS -> tool_calls
  step: RunStepType.TOOL_CALLS -> tool_calls
  step: RunStepType.TOOL_CALLS -> tool_calls



ASSISTANT: Your identity has been verified, and I've submitted a refill request for your expired cholesterol medication, atorvastatin 20mg. You can expect it to be ready in approximately 2 days. If you need anything else, feel free to ask!


---
## Step 4 — Clean up

Agents are persistent resources — delete the demo agent so the project stays tidy (the seeded production agents are untouched).


In [6]:
if agent is not None:
    try:
        agents.delete_agent(agent.id)
        print('Deleted demo agent', agent.id)
    except Exception as e:
        print('[warn] cleanup:', e)
else:
    print('Nothing to clean up.')


Deleted demo agent asst_wc9V4vbBb5WsA2otwwIqxPOG


---
## Takeaways

- A **persistent agent** turns a one-shot model into a multi-step worker that plans + calls tools itself.
- Your functions stay **local business logic**; Foundry handles orchestration, threading and retries.
- Every step is **traced** (Step 0) — open the Foundry Tracing tab to replay the tool chain.
- This is the biggest build-vs-buy gap: Foundry ships agent orchestration as a **managed service**.

*← The Decision Advisor (Lab 09) routes the `needs_agents` flag here.*
